# 从真实实验数据突变列表生成Stage 1数据集

### 目的
该脚本旨在根据一个包含E蛋白突变位点、`Log2Expression`和`CCK8`实验数值的Excel文件，生成一个用于模型训练的 `.pkl` 格式数据集。此版本直接从表格中读取WT（野生型）数据，不再手动添加。

### 流程
1.  **配置设置**：设置所有文件路径、模型设备和输出文件名。
2.  **加载模型**：加载预训练的ESM-3模型。
3.  **加载参考数据**：读取参考的 `E.fasta` 序列和 `E.pdb` 结构，并进行一次性编码。
4.  **处理突变数据**：
    a. 读取 `250930_M_Mut.xlsx` 文件。
    b. 遍历每一行数据（包括WT行），动态生成对应的蛋白质序列。
    c. 对每个序列进行编码。
    d. 将样本ID(`Mutation`)、`Log2Expression`、`CCK8`以及编码后的序列和结构信息，按照指定格式组装成字典。
5.  **保存结果**：将所有处理好的样本数据保存为一个单独的PKL文件。

### 1. 导入所需库

In [1]:
import os
import re
import pickle
import torch
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm

# 导入ESM相关模块
from esm.models.esm3 import ESM3
from esm.sdk.api import ESMProtein

### 2. 参数配置
请在此单元格中确认所有路径和参数。

In [2]:
# ----- 路径和参数配置 -----

# 1. 输入：包含突变信息的Excel文件路径
MUTATION_DATA_PATH = "/data2/zhoukaitao/01evoModel/dataset/251124_E_Mut_R2/251124_E_Mut_R2.xlsx"

# 2. 输入：参考序列FASTA文件路径
REF_FASTA_PATH = "./fasta/E.fasta"

# 3. 输入：参考结构PDB文件路径
REF_PDB_PATH = "./pdb/E.pdb"

# 4. 输出：最终生成的PKL文件路径
OUTPUT_PKL_PATH = "/data2/zhoukaitao/01evoModel/dataset/251124_E_Mut_R2/E_Mut_R2.pkl"

# 5. 指定用于计算的设备 (例如 'cuda:0' 或 'cpu')
DEVICE = torch.device("cuda:3" if torch.cuda.is_available() else "cpu")

# 确保输出目录存在
os.makedirs(os.path.dirname(OUTPUT_PKL_PATH), exist_ok=True)

print(f"突变数据源: {MUTATION_DATA_PATH}")
print(f"参考FASTA: {REF_FASTA_PATH}")
print(f"参考PDB: {REF_PDB_PATH}")
print(f"输出PKL文件: {OUTPUT_PKL_PATH}")
print(f"计算设备: {DEVICE}")

突变数据源: /data2/zhoukaitao/01evoModel/dataset/251124_E_Mut_R2/251124_E_Mut_R2.xlsx
参考FASTA: ./fasta/E.fasta
参考PDB: ./pdb/E.pdb
输出PKL文件: /data2/zhoukaitao/01evoModel/dataset/251124_E_Mut_R2/E_Mut_R2.pkl
计算设备: cuda:3


### 3. 辅助函数定义

In [3]:
def read_single_fasta(file_path):
    """读取一个只包含单条序列的FASTA文件。"""
    with open(file_path, 'r') as f:
        lines = f.readlines()
    return "".join([line.strip() for line in lines if not line.startswith('>')])

def apply_mutation(sequence, mutation_str):
    """
    根据一个或多个由'-'连接的突变字符串（例如 'T9I' 或 'T9I-T11A'）修改原始序列。
    返回修改后的新序列。
    """
    # 将原始序列转换为列表，以便进行修改
    mutated_seq_list = list(sequence)
    
    # 通过'-'分割字符串，以处理单个或多个突变
    individual_mutations = mutation_str.split('-')
    
    # 遍历列表中的每一个独立突变
    for single_mutation in individual_mutations:
        # 使用正则表达式解析单个突变
        match = re.match(r'([A-Z])([0-9]+)([A-Z])', single_mutation, re.IGNORECASE)
        if not match:
            print(f"  [警告] 无法解析突变格式中的一部分: '{single_mutation}' (在 '{mutation_str}' 中)，将跳过整个样本。")
            return None # 如果任何一部分格式错误，则整个样本无效
        
        original_aa, position_str, mutated_aa = match.groups()
        position = int(position_str) - 1 # 转换为0-based索引

        # 安全性检查
        if position < 0 or position >= len(mutated_seq_list):
            print(f"  [警告] 突变 '{single_mutation}' 的位置 {position+1} 超出序列长度 {len(mutated_seq_list)} (在 '{mutation_str}' 中)，将跳过整个样本。")
            return None
        
        # 检查原始氨基酸是否匹配。注意：这里检查的是序列的当前状态（可能已被前一个突变修改）
        if mutated_seq_list[position].upper() != original_aa.upper():
            print(f"  [警告] 在处理 '{mutation_str}' 时，突变 '{single_mutation}' 的原始氨基酸不匹配：期望 {original_aa}，实际为 {mutated_seq_list[position]}。仍将执行替换。")
        
        # 应用当前突变
        mutated_seq_list[position] = mutated_aa.upper()
        
    # 将列表重新组合成字符串并返回
    return "".join(mutated_seq_list)

### 4. 加载模型与参考数据

In [4]:
print("正在加载 ESM-3 模型...")
model = ESM3.from_pretrained("esm3_sm_open_v1", DEVICE)
model.eval()
print("ESM-3 模型加载完成。")

print("\n正在加载并编码参考序列和结构...")
try:
    ref_sequence = read_single_fasta(REF_FASTA_PATH)
    ref_protein_seq_obj = ESMProtein(sequence=ref_sequence)
    with torch.no_grad():
        encoded_ref_seq = model.encode(ref_protein_seq_obj).sequence.cpu().numpy()
    
    ref_protein_pdb_obj = ESMProtein.from_pdb(REF_PDB_PATH)
    with torch.no_grad():
        encoded_pdb_structure = model.encode(ref_protein_pdb_obj).structure.cpu().numpy()
    
    aligned_pro_dict = {
        'seq_t': encoded_ref_seq,
        'structure_t': encoded_pdb_structure
    }
    print("参考数据编码完成。")
except FileNotFoundError as e:
    print(f"[错误] 找不到参考文件: {e}，请检查路径配置。脚本将终止。")
    raise

正在加载 ESM-3 模型...
ESM-3 模型加载完成。

正在加载并编码参考序列和结构...
参考数据编码完成。


### 5. 主处理流程

In [5]:
try:
    # 使用 usecols 参数只读取我们需要的列
    df_mutations = pd.read_excel(MUTATION_DATA_PATH, usecols=['Mutation', 'Log2Expression', 'CCK8', 'Activation'])       #['Mutation', 'Log2Expression', 'CCK8', 'Activation','TP_CCK8'])
    # 剔除任何一行中存在NaN值的情况
    df_mutations.dropna(inplace=True)
    print(f"成功读取并清洗突变数据文件，共 {len(df_mutations)} 条有效记录。")
except FileNotFoundError:
    print(f"[错误] 突变数据文件未找到: {MUTATION_DATA_PATH}")
    raise

all_results_for_pkl = []

# 遍历Excel中的每一条记录
for _, row in tqdm(df_mutations.iterrows(), total=len(df_mutations), desc="处理样本"):
    mutation_id = row['Mutation']
    expression_val = row['Log2Expression']
    cck8_val = row['CCK8']
    activation_val = row['Activation']
    # cck8_val = row['TP_CCK8']
    
    # a. 根据ID决定序列 (WT或突变体)
    if mutation_id == 'WT':
        current_seq = ref_sequence
    else:
        current_seq = apply_mutation(ref_sequence, mutation_id)
        if current_seq is None: # 如果突变格式有误或位置超限，则跳过
            continue
    
    # b. 编码当前序列
    try:
        with torch.no_grad():
            seq_obj = ESMProtein(sequence=current_seq)
            encoded_seq = model.encode(seq_obj).sequence.cpu().numpy()
    except Exception as e:
        print(f"  [错误] 编码序列 '{mutation_id}' 时失败: {e}")
        continue
    
    # c. 组装样本字典
    sample_record = {
        'id': mutation_id,
        # 'days':activation_val,
        'Expression': expression_val,
        'CCK8': cck8_val,
        'Activation':activation_val,
        'E': {
            'seq_t': encoded_seq,
            'structure_t': encoded_pdb_structure
        },
        'aligned_E': aligned_pro_dict
    }
    
    all_results_for_pkl.append(sample_record)

# 保存最终结果
if all_results_for_pkl:
    with open(OUTPUT_PKL_PATH, 'wb') as f_out:
        pickle.dump(all_results_for_pkl, f_out)
    print(f"\n处理完成！已将 {len(all_results_for_pkl)} 条记录保存至: {OUTPUT_PKL_PATH}")
else:
    print("\n未能成功处理任何记录，没有生成PKL文件。")

成功读取并清洗突变数据文件，共 29 条有效记录。


处理样本:   0%|          | 0/29 [00:00<?, ?it/s]


处理完成！已将 29 条记录保存至: /data2/zhoukaitao/01evoModel/dataset/251124_E_Mut_R2/E_Mut_R2.pkl


### 6. (可选) 验证输出文件

In [6]:
if os.path.exists(OUTPUT_PKL_PATH):
    print(f"正在加载并验证文件: {OUTPUT_PKL_PATH}")
    with open(OUTPUT_PKL_PATH, 'rb') as f:
        loaded_data = pickle.load(f)
    
    if isinstance(loaded_data, list) and len(loaded_data) > 0:
        print(f"文件加载成功，共包含 {len(loaded_data)} 条记录。")
        
        # 检查第一条记录的结构 (应该是WT)
        first_item = loaded_data[0]
        print("\n--- 检查第一条记录的结构 ---")
        print(f"ID: {first_item.get('id')}")
        print(f"Expression: {first_item.get('Expression')}")
        print(f"CCK8: {first_item.get('CCK8')}")
        print(f"包含的顶级键: {list(first_item.keys())}")
        
        if 'E' in first_item and 'aligned_E' in first_item:
            print("E 和 aligned_E 键存在，结构正确。")
        else:
            print("错误：数据结构不完整！")
    else:
        print("PKL文件为空或格式不正确。")
else:
    print(f"错误: 找不到要验证的PKL文件: {OUTPUT_PKL_PATH}")

正在加载并验证文件: /data2/zhoukaitao/01evoModel/dataset/251124_E_Mut_R2/E_Mut_R2.pkl
文件加载成功，共包含 29 条记录。

--- 检查第一条记录的结构 ---
ID: WT
Expression: 0.0
CCK8: 0.519788017380802
包含的顶级键: ['id', 'Expression', 'CCK8', 'Activation', 'E', 'aligned_E']
E 和 aligned_E 键存在，结构正确。


In [7]:
loaded_data

[{'id': 'WT',
  'Expression': 0.0,
  'CCK8': 0.519788017380802,
  'Activation': 0.739674593241552,
  'E': {'seq_t': array([ 0, 20, 19,  8, 18,  7,  8,  9,  9, 11,  6, 11,  4, 12,  7, 17,  8,
           7,  4,  4, 18,  4,  5, 18,  7,  7, 18,  4,  4,  7, 11,  4,  5, 12,
           4, 11,  5,  4, 10,  4, 23,  5, 19, 23, 23, 17, 12,  7, 17,  7,  8,
           4,  7, 15, 14,  8, 18, 19,  7, 19,  8, 10,  7, 15, 17,  4, 17,  8,
           8, 10,  7, 14, 13,  4,  4,  7,  2]),
   'structure_t': array([4098, 2048,  264, 2439,  137,  264, 1197, 2048, 3056,  264,  264,
           137,  264, 3961, 2048, 1197, 2056, 1476, 1197,  588,  123,  588,
           588, 1450,  264, 1476, 1450,  588, 1476, 2048, 1197,  588,  588,
          2048, 1197,  588,  588, 1476, 1197,  588,  588, 1476,  588,  588,
          1197,  588, 1197, 1476, 1197,  445, 2983, 3407, 2048,  137, 2156,
          1197,  588, 1197, 2048,  588,  588,  987, 1476,  588, 1197, 2048,
          1197,  123, 1197, 2439, 1892, 2093, 1265,  264

In [8]:
with open("/data2/zhoukaitao/01evoModel/dataset/250930_E_Mut/E_activate_fold/1_train.pkl", 'rb') as f:
        loaded_data = pickle.load(f)
len(loaded_data)

62

In [9]:
with open("/data2/zhoukaitao/01evoModel/dataset/250930_E_Mut/E_activate_fold/1_val.pkl", 'rb') as f:
        loaded_data_val = pickle.load(f)
len(loaded_data_val)

21

In [10]:
loaded_data_val


[{'id': 'T30A',
  'days': 1.80172413872628,
  'Expression': 0.305900529552289,
  'CCK8': 1.53001865676733,
  'Activation': 1.80172413872628,
  'E': {'seq_t': array([ 0, 20, 19,  8, 18,  7,  8,  9,  9, 11,  6, 11,  4, 12,  7, 17,  8,
           7,  4,  4, 18,  4,  5, 18,  7,  7, 18,  4,  4,  7,  5,  4,  5, 12,
           4, 11,  5,  4, 10,  4, 23,  5, 19, 23, 23, 17, 12,  7, 17,  7,  8,
           4,  7, 15, 14,  8, 18, 19,  7, 19,  8, 10,  7, 15, 17,  4, 17,  8,
           8, 10,  7, 14, 13,  4,  4,  7,  2]),
   'structure_t': array([4098, 2048,  264, 2439,  137,  264, 1197, 2048, 3056,  264,  264,
           137,  264, 3961, 2048, 1197, 2056, 1476, 1197,  588,  123,  588,
           588, 1450,  264, 1476, 1450,  588, 1476, 2048, 1197,  588,  588,
          2048, 1197,  588,  588, 1476, 1197,  588,  588, 1476,  588,  588,
          1197,  588, 1197, 1476, 1197,  445, 2983, 3407, 2048,  137, 2156,
          1197,  588, 1197, 2048,  588,  588,  987, 1476,  588, 1197, 2048,
          1197

In [11]:
loaded_data

[{'id': 'S3K',
  'days': 1.28134796294801,
  'Expression': -3.35259902848571,
  'CCK8': 1.76046141918175,
  'Activation': 1.28134796294801,
  'E': {'seq_t': array([ 0, 20, 19, 15, 18,  7,  8,  9,  9, 11,  6, 11,  4, 12,  7, 17,  8,
           7,  4,  4, 18,  4,  5, 18,  7,  7, 18,  4,  4,  7, 11,  4,  5, 12,
           4, 11,  5,  4, 10,  4, 23,  5, 19, 23, 23, 17, 12,  7, 17,  7,  8,
           4,  7, 15, 14,  8, 18, 19,  7, 19,  8, 10,  7, 15, 17,  4, 17,  8,
           8, 10,  7, 14, 13,  4,  4,  7,  2]),
   'structure_t': array([4098, 2048,  264, 2439,  137,  264, 1197, 2048, 3056,  264,  264,
           137,  264, 3961, 2048, 1197, 2056, 1476, 1197,  588,  123,  588,
           588, 1450,  264, 1476, 1450,  588, 1476, 2048, 1197,  588,  588,
          2048, 1197,  588,  588, 1476, 1197,  588,  588, 1476,  588,  588,
          1197,  588, 1197, 1476, 1197,  445, 2983, 3407, 2048,  137, 2156,
          1197,  588, 1197, 2048,  588,  588,  987, 1476,  588, 1197, 2048,
          1197,